# Visión por computador - Practica 2

En esta competición, el objetivo principal es alcanzar el F1-score más alto posible, demostrando
tu destreza en la construcción y evaluación de modelos. El F1-score es una métrica fundamental
que mide la precisión y la exhaustividad de tus predicciones, lo que significa que encontrarás un
equilibrio perfecto entre la capacidad de acertar y evitar errores.
La participación en la competición únicamente será posible en grupos de 3-4. El dataset para la
práctica ha sido proporcionado en la entrega. Las características de la clasificación o qué objetos
se clasifican se deja a elección de los participantes (se recomienda evitar soluciones triviales, ex:
tiene color rojo-no tiene color rojo). Cada tipo de clasificación se considerará una categoría
distinta. En caso de necesitar reorganizar las imágenes dentro de cada conjunto de datos (sin
mover imágenes entre particiones) para realizar otro tipo de clasificación, se deberá entregar el
conjunto de datos completo ya reorganizado. En ningún caso se podrán eliminar imágenes de
cualquiera de las tres particiones. Se puede utilizar data augmentation, redimensionamiento y
otras modificaciones.
En el improbable caso de un empate en el F1-score, no te preocupes, analizaremos las tasas de
falsos positivos y falsos negativos para determinar al ganador. Estas tasas son críticas para
evaluar cuán efectivo es tu modelo en la detección de errores y cómo maneja los casos negativos
y positivos de manera precisa.

## Fase 1 - Carga de datos

El dataset `cards/` ya viene dividido en tres particiones (`train`, `valid`, `test`) y cada partición tiene una carpeta por clase (53 clases: 52 cartas + joker). Las imágenes son RGB de 224x224.

Como la estructura es `particion/clase/imagen.jpg`, se puede cargar directamente con `torchvision.datasets.ImageFolder`, que asigna una etiqueta a cada subcarpeta.

En esta fase **no** se hace preprocesado (solo se pasa la imagen a tensor). El redimensionado, la normalización y el data augmentation se harán en la fase 2.

In [ ]:
import os
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", device)

### Rutas del dataset
La carpeta `cards` se busca automáticamente de forma relativa, así que no hace falta cambiar nada aunque el kernel arranque en otra carpeta. En Colab, basta con descomprimir `cards` en la carpeta de trabajo.

In [ ]:
SPLITS = ["train", "valid", "test"]

# Buscamos la carpeta "cards" de forma relativa: en la carpeta actual, en sus
# padres y en las subcarpetas de la practica (local "Practica 2" o repo "practica2").
# Asi funciona aunque Jupyter/VS Code arranque el kernel desde otra carpeta.
def buscar_dataset(nombre="cards"):
    actual = os.path.abspath(os.getcwd())
    bases = [actual]
    while os.path.dirname(bases[-1]) != bases[-1]:
        bases.append(os.path.dirname(bases[-1]))
    for base in bases:
        for sub in ["", "Practica 2", "practica2"]:
            candidata = os.path.join(base, sub, nombre)
            if all(os.path.isdir(os.path.join(candidata, s)) for s in SPLITS):
                return os.path.relpath(candidata)
    raise FileNotFoundError(f"No se encuentra la carpeta '{nombre}' con {SPLITS} desde {actual}")

DATA_DIR = buscar_dataset()
print("Dataset en:", DATA_DIR)

### Exploración del dataset
Contamos cuántas imágenes hay en cada clase y partición, para ver si las clases están balanceadas.

In [ ]:
conteo = {}
for split in SPLITS:
    ruta = os.path.join(DATA_DIR, split)
    clases = sorted(os.listdir(ruta))
    conteo[split] = {c: len(os.listdir(os.path.join(ruta, c))) for c in clases}
    total = sum(conteo[split].values())
    print(f"{split:5s}: {len(clases)} clases, {total} imágenes")

# Las tres particiones deben tener exactamente las mismas clases
assert conteo["train"].keys() == conteo["valid"].keys() == conteo["test"].keys()

In [ ]:
clases = list(conteo["train"].keys())
plt.figure(figsize=(14, 4))
plt.bar(clases, [conteo["train"][c] for c in clases])
plt.xticks(rotation=90, fontsize=7)
plt.ylabel("Nº imágenes")
plt.title("Imágenes por clase en train")
plt.tight_layout()
plt.show()

print("Mínimo en train:", min(conteo["train"].values()), "| Máximo en train:", max(conteo["train"].values()))
print("Imágenes por clase en valid/test:", set(conteo["valid"].values()), set(conteo["test"].values()))

### Carga con ImageFolder y DataLoaders
De momento la única transformación es `ToTensor()`, que convierte la imagen a tensor `[C, H, W]` con valores en `[0, 1]`.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
])

train_dataset = datasets.ImageFolder(os.path.join(DATA_DIR, "train"), transform=transform)
val_dataset   = datasets.ImageFolder(os.path.join(DATA_DIR, "valid"), transform=transform)
test_dataset  = datasets.ImageFolder(os.path.join(DATA_DIR, "test"),  transform=transform)

# Comprobamos que el índice de cada clase es el mismo en las tres particiones
assert train_dataset.class_to_idx == val_dataset.class_to_idx == test_dataset.class_to_idx

class_names = train_dataset.classes
num_classes = len(class_names)
print("Nº clases:", num_classes)
print("Train:", len(train_dataset), "| Valid:", len(val_dataset), "| Test:", len(test_dataset))

In [ ]:
BATCH_SIZE = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

images, labels = next(iter(train_loader))
print("Forma del batch de imágenes:", images.shape)   # [batch, canales, alto, ancho]
print("Forma del batch de etiquetas:", labels.shape)
print("Rango de valores:", images.min().item(), "-", images.max().item())

### Visualización de algunas imágenes

In [ ]:
plt.figure(figsize=(12, 6))
for i in range(8):
    plt.subplot(2, 4, i + 1)
    img = images[i].permute(1, 2, 0).numpy()   # [C,H,W] -> [H,W,C] para matplotlib
    plt.imshow(img)
    plt.title(class_names[labels[i]], fontsize=9)
    plt.axis("off")
plt.tight_layout()
plt.show()